# Extra: The Unreasonable Effectiveness of the Agent Loop

This lab crystallises the single most important concept in the course.

## What is an Agent?

Three popular definitions:

1. *"AI systems that can do work for you independently"* — Sam Altman
2. *"A system in which an LLM controls the workflow"* — Anthropic
3. *"An LLM that runs tools in a loop to achieve a goal"* — emerging consensus

The third definition is the most concrete and useful. Let's make it real by building a **TODO agent** from scratch — no frameworks, just the loop.

In [ ]:
# rich makes terminal output much nicer
# If missing: uv add rich

from rich.console import Console
from dotenv import load_dotenv
from google import genai
from google.genai import types
import json

load_dotenv(override=True)
client = genai.Client()
console = Console()

In [ ]:
def show(text: str):
    """Display text with Rich formatting (falls back to plain print)."""
    try:
        console.print(text)
    except Exception:
        print(text)

## The TODO tools

We give the agent two tools:
- `create_todos` — add items to a list
- `mark_complete` — tick off an item

The agent will plan its approach using these tools, then execute each step.

In [ ]:
todos: list[str] = []
completed: list[bool] = []

In [ ]:
def get_todo_report() -> str:
    """Return a formatted string of all todos with completion status."""
    if not todos:
        return "No todos yet."
    result = ""
    for i, todo in enumerate(todos):
        if completed[i]:
            result += f"Todo #{i + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{i + 1}: {todo}\n"
    show(result)
    return result


def create_todos(descriptions: list[str]) -> str:
    """Add new todos from a list of descriptions. Returns the updated todo list.

    Args:
        descriptions: List of todo item descriptions to add
    """
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()


def mark_complete(index: int, completion_notes: str) -> str:
    """Mark a todo as complete and return the updated list.

    Args:
        index: 1-based index of the todo to complete
        completion_notes: Brief notes about how this step was completed (Rich markup ok)
    """
    if 1 <= index <= len(todos):
        completed[index - 1] = True
        console.print(completion_notes)
    else:
        return "No todo at that index."
    return get_todo_report()

In [ ]:
# Quick sanity check
todos, completed = [], []
create_todos(["Buy groceries", "Finish this lab", "Eat a banana"])
mark_complete(1, "[green]Done! Got apples, bananas, and oat milk.[/green]")

## The Agent Loop

This is the core pattern. Read it carefully — you'll see it everywhere:

```
while True:
    response = LLM(messages, tools)
    
    if no tool calls:
        break  # agent is done
    
    execute tool calls
    add results back to messages
```

The LLM decides when to stop. It runs in a loop until it produces a final answer with no tool calls.

In [ ]:
tool_config = types.GenerateContentConfig(
    tools=[create_todos, mark_complete]
)


def agent_loop(system_message: str, user_message: str):
    """Run the agent loop until it produces a final answer."""
    contents = [
        types.Content(role="user", parts=[types.Part(text=user_message)])
    ]

    config = types.GenerateContentConfig(
        system_instruction=system_message,
        tools=[create_todos, mark_complete]
    )

    while True:
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=contents,
            config=config
        )

        # No tool calls — the agent has a final answer
        if not response.function_calls:
            break

        # Add model's response to the conversation history
        contents.append(response.candidates[0].content)

        # Execute each tool call and collect results
        tool_parts = []
        for fc in response.function_calls:
            tool_fn = globals().get(fc.name)
            result = tool_fn(**dict(fc.args)) if tool_fn else {"error": "unknown tool"}
            tool_parts.append(
                types.Part.from_function_response(
                    name=fc.name,
                    response={"result": str(result)}
                )
            )

        # Add tool results back into the conversation
        contents.append(types.Content(role="user", parts=tool_parts))

    show(response.text)

In [ ]:
system_message = """
You solve problems step-by-step using your todo tools.

Process:
1. Use create_todos to plan a list of steps
2. Work through each step in order, calling mark_complete after each one
3. Reply with your final answer in Rich console markup

If any quantity isn't given, estimate it. Do not ask clarifying questions.
"""

user_message = """
A train leaves Nairobi at 2:00pm travelling at 80 km/h toward Mombasa.
Another train leaves Mombasa at 3:00pm travelling toward Nairobi at 100 km/h.
The distance between the cities is 480 km.
When and where do they meet?
"""

In [ ]:
todos, completed = [], []
agent_loop(system_message, user_message)

## What just happened?

The agent:
1. **Planned** — called `create_todos` to lay out its approach
2. **Executed** — worked through each step, calling `mark_complete` as it went
3. **Answered** — once all steps were done, it synthesised a final answer

This is the agent loop. Every major agentic framework (LangGraph, CrewAI, AutoGen) is built on this same principle.

The power: the **LLM decides how many steps to take** and what tools to call. You don't hard-code the workflow — the model reasons its way through it.

In [ ]:
# Try a harder problem!

hard_problem = """
I want to save enough money to buy a laptop worth KES 150,000.
I currently save KES 8,000 per month.
I have KES 25,000 saved already.
But from next month I'm getting a 15% raise, so my savings will increase.
When will I have enough?
"""

todos, completed = [], []
agent_loop(system_message, hard_problem)

## Exercise

**Build an agent loop from scratch yourself.**

Create a new notebook and implement the agent loop from first principles — referring back to this notebook as needed, but typing it yourself.

Suggested challenges:
- Add a `research_topic(topic)` tool that calls the LLM to explain something, and have the agent use it while solving a multi-part question
- Add a `calculate(expression)` tool that uses Python's `eval()` for maths
- Create an agent that plans a 5-day study schedule for a topic you choose

Save your work in `exercises/your_name/`!